In [1]:
!pip install -q open3d
!pip install -q torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.4.1+cu121.html
!pip install -q timm pyvista

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 89.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 87.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 104.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 kB 15.6 MB/s eta 0:00:00


In [2]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# --- CONFIGURATION ---
CONFIG = {
    "ROOT_3D": '/kaggle/input/datasets/klein2111/scannet-3d/scannet_3d/train',
    "ROOT_2D": '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d',
    "NUM_POINTS": 65536,    
    "BATCH_SIZE": 4,        
    "NUM_CLASSES": 20,      
    "PRETRAIN_EPOCHS": 3,   
    "FINETUNE_EPOCHS": 40,  # 40 epochs is perfect for an augmented dataset
    "LR_PRETRAIN": 0.004,
    "LR_FINETUNE": 0.001,
    "DEVICE": torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

# --- ADVANCED FOCAL LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, ignore_index=255):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.ce = nn.CrossEntropyLoss(weight=alpha, ignore_index=ignore_index, reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss) # Probability of the correct class
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        # We average the loss only over valid (non-ignored) points
        valid_mask = (targets != self.ignore_index).float()
        return (focal_loss * valid_mask).sum() / valid_mask.sum().clamp(min=1.0)

In [3]:
class ScanNetAugmentedDataset(Dataset):
    def __init__(self, root_3d, root_2d, num_points=65536, augment=True):
        self.root_3d = root_3d
        self.root_2d = root_2d
        self.num_points = num_points
        self.augment = augment
        self.file_list = sorted([f for f in os.listdir(root_3d) if f.endswith('.pth')])

    def __len__(self):
        return len(self.file_list)
        
    def apply_augmentation(self, points):
        # 1. Random Z-axis Rotation
        theta = np.random.uniform(0, 2 * np.pi)
        rot_matrix = np.array([
            [np.cos(theta), -np.sin(theta), 0],
            [np.sin(theta),  np.cos(theta), 0],
            [0,              0,             1]
        ])
        points = np.dot(points, rot_matrix)
        
        # 2. Random Jitter (Noise)
        noise = np.random.normal(0, 0.01, size=points.shape)
        points += noise
        return points

    def __getitem__(self, idx):
        file_name = self.file_list[idx]
        scene_id = file_name.replace('_vh_clean_2.pth', '')
        
        pc_path = os.path.join(self.root_3d, file_name)
        data_3d = torch.load(pc_path, weights_only=False)
        
        if isinstance(data_3d, tuple):
            points = data_3d[0]
            labels = data_3d[2] if len(data_3d) > 2 else data_3d[1] 
        elif isinstance(data_3d, dict):
            points = data_3d['coord']
            labels = data_3d.get('semantic_gt', data_3d.get('label', np.zeros(len(points))))
        else:
            points = data_3d[:, :3]
            labels = np.zeros(len(points))
            
        points = np.array(points) if torch.is_tensor(points) else points
        labels = np.array(labels) if torch.is_tensor(labels) else labels

        if len(points) >= self.num_points:
            s_idx = np.random.choice(len(points), self.num_points, replace=False)
        else:
            s_idx = np.random.choice(len(points), self.num_points, replace=True)
            
        points = points[s_idx]
        labels = labels[s_idx]
        
        # Apply 3D Augmentation during training
        if self.augment:
            points = self.apply_augmentation(points)
            
        img_path = os.path.join(self.root_2d, scene_id, "color", "0.jpg")
        if os.path.exists(img_path):
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (518, 518))
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        else:
            image = torch.zeros((3, 518, 518))

        return {
            "points": torch.as_tensor(points).float(), 
            "labels": torch.as_tensor(labels).long(),
            "image": image
        }

In [4]:
def load_2d_teacher():
    model = timm.create_model('vit_small_patch14_dinov2', pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    model.eval()
    return model

class ConcertoStudent(nn.Module):
    def __init__(self, output_dim=384): 
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(3, 128), nn.ReLU(),
            nn.Linear(128, 384)
        )
        self.predictor = nn.Linear(384, output_dim)

    def forward(self, x):
        features = self.backbone(x)
        return self.predictor(features)

class SpatialSynergyNet(nn.Module):
    def __init__(self, pretrained_backbone, num_classes=20):
        super().__init__()
        self.backbone = pretrained_backbone
        
        # UNFREEZE: We train the entire backbone now for max accuracy
        for param in self.backbone.parameters():
            param.requires_grad = True
            
        # Deep, uncompressed Multi-Layer Perceptron Head
        self.task_head = nn.Sequential(
            nn.Linear(384, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes) 
        )

    def forward(self, x):
        return self.task_head(self.backbone(x))

In [5]:
def run_ultra_pipeline():
    # Use the new Augmented Dataset
    dataset = ScanNetAugmentedDataset(CONFIG['ROOT_3D'], CONFIG['ROOT_2D'], augment=True)
    loader = DataLoader(dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=True)
    print(f"Data Loaded: {len(dataset)} scenes ready for Ultra Training.")

    student = ConcertoStudent().to(CONFIG['DEVICE'])
    teacher = load_2d_teacher().to(CONFIG['DEVICE'])
    
    # --- PHASE 1: PRE-TRAINING ---
    print("\n--- PHASE 1: PRE-TRAINING ---")
    optimizer_pre = optim.AdamW(student.parameters(), lr=CONFIG['LR_PRETRAIN'])
    
    for epoch in range(CONFIG['PRETRAIN_EPOCHS']):
        student.train()
        total_loss = 0
        pbar = tqdm(loader, desc=f"Pretrain Epoch {epoch+1}")
        for batch in pbar:
            points, images = batch['points'].to(CONFIG['DEVICE']), batch['image'].to(CONFIG['DEVICE'])
            optimizer_pre.zero_grad()
            s_global = student(points).mean(dim=1) 
            with torch.no_grad():
                t_feat = teacher(images)   
            loss = (1 - F.cosine_similarity(s_global, t_feat, dim=1)).mean()
            loss.backward()
            optimizer_pre.step()
            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    # --- PHASE 2: FULL FINE-TUNING WITH FOCAL LOSS ---
    print("\n--- PHASE 2: ADVANCED FINE-TUNING ---")
    final_model = SpatialSynergyNet(student.backbone, num_classes=CONFIG['NUM_CLASSES']).to(CONFIG['DEVICE'])
    
    # Optimizer handles the entire network now
    optimizer_full = optim.AdamW(final_model.parameters(), lr=CONFIG['LR_FINETUNE'], weight_decay=1e-4)
    
    # Base class weights (to help Focal Loss start in the right direction)
    base_weights = torch.tensor([
        0.5, 0.5, 2.0, 2.0, 3.0, 2.5, 2.0, 1.5, 1.5, 2.5, 
        4.0, 3.0, 3.0, 2.5, 3.0, 4.0, 4.0, 4.0, 3.0, 1.0
    ]).to(CONFIG['DEVICE'])

    # Initialize our custom Focal Loss
    criterion = FocalLoss(alpha=base_weights, gamma=2.0, ignore_index=255)
    
    for epoch in range(CONFIG['FINETUNE_EPOCHS']):
        final_model.train()
        total_loss = 0
        pbar = tqdm(loader, desc=f"Finetune Epoch {epoch+1}")
        for batch in pbar:
            points, labels = batch['points'].to(CONFIG['DEVICE']), batch['labels'].to(CONFIG['DEVICE'])
            labels[(labels < 0) | (labels >= CONFIG['NUM_CLASSES'])] = 255
            
            optimizer_full.zero_grad()
            preds = final_model(points).permute(0, 2, 1) 
            
            loss = criterion(preds, labels)
            loss.backward()
            optimizer_full.step()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        print(f"Epoch {epoch+1} Avg Loss: {total_loss/len(loader):.4f}")

    # SAVE THE MASTER MODEL
    torch.save(final_model.state_dict(), "SpatialSynergyNet_ULTRA.pth")
    print("\nSUCCESS: Model saved as 'SpatialSynergyNet_ULTRA.pth'")

run_ultra_pipeline()

Data Loaded: 1201 scenes ready for Ultra Training.


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]


--- PHASE 1: PRE-TRAINING ---


Pretrain Epoch 3: 100%|██████████| 301/301 [02:00<00:00,  2.50it/s, loss=0.5766]



--- PHASE 2: ADVANCED FINE-TUNING ---


...

Epoch 38 Avg Loss: 2.5471


Finetune Epoch 39: 100%|██████████| 301/301 [02:05<00:00,  2.40it/s, loss=1.4211]


Epoch 39 Avg Loss: 2.5365


Finetune Epoch 40: 100%|██████████| 301/301 [02:05<00:00,  2.41it/s, loss=2.6969]

Epoch 40 Avg Loss: 2.5306

SUCCESS: Model saved as 'SpatialSynergyNet_ULTRA.pth'
